# CUDA

CUDA is NVIDIA's parallel computing platform and programming model for NVIDIA GPUs. It allows applications such as PyTorch to use NVIDIA GPUs for general-purpose computation.

---

# 1. Allocator

The **CUDA Caching Allocator** is an internal component of PyTorch that manages GPU memory.

It works automatically in the background when tensors need GPU memory. Instead of constantly requesting and releasing memory directly from CUDA, the allocator keeps a pool of memory that can be reused.

## 1.1 Segments

**Segments** are large regions of GPU memory obtained by the allocator from CUDA.

The allocator manages these segments and organizes them into smaller blocks.

```text
GPU
└── Segment
```

## 1.2 Blocks

**Blocks** are smaller regions inside a segment that the allocator manages.

Blocks can be either:

* **Allocated** → currently being used by a tensor
* **Free** → available for future allocations

```text
Segment
├── Block → Tensor
├── Block → Free
└── Block → Tensor
```

---

# 2. Workflow

## 2.1 Splitting

**Splitting** happens when the allocator divides a large free block into smaller blocks to satisfy an allocation.

```text
Before:

┌──────────────────────┐
│     Free: 100 MB     │
└──────────────────────┘

After splitting:

┌──────────┬───────────┐
│ 30 MB    │ 70 MB     │
│ Tensor   │ Free      │
└──────────┴───────────┘
```

In this example:

**1 block → 2 blocks**

## 2.2 Merging

**Merging** (or coalescing) happens when adjacent free blocks are combined into a larger free block.

```text
Before:

┌──────────┬───────────┐
│ 30 MB    │ 70 MB     │
│ Free     │ Free      │
└──────────┴───────────┘

After merging:

┌──────────────────────┐
│     Free: 100 MB     │
└──────────────────────┘
```

In this example:

**2 blocks → 1 block**

## 2.3 Fragmentation

**Fragmentation** happens when available free memory is divided into separate blocks instead of being available as one sufficiently large contiguous region.

```text
┌────────┬────────┬────────┬────────┐
│ 20 MB  │ 30 MB  │ 20 MB  │ 30 MB  │
│ Free   │ Used   │ Free   │ Used   │
└────────┴────────┴────────┴────────┘
```

There is **40 MB of free memory in total**, but the largest contiguous free block is only **20 MB**.

Therefore, an allocation requiring **30 MB** cannot use those two separate 20 MB blocks as if they were one 40 MB block.

This is called **memory fragmentation**.

Merging can help reduce fragmentation when free blocks are adjacent.


---

# 3. CUDA Utilities (`torch.cuda`)

`torch.cuda` is a utility module for interacting with NVIDIA GPUs (not a model-building module like `torch.nn`).

So this module we can control the GPUs using the torch utilities

1. `torch.cuda.is_available()`
   Checks if a GPU with CUDA is available.

2. `torch.cuda.device_count()`
   Returns how many GPUs are available.

3. `torch.cuda.current_device()`
   Returns the index of the GPU currently in use.

4. `torch.cuda.get_device_name(index)`
   Returns the name of the GPU at the given index.

> More advanced args

5. `torch.cuda.memory_allocated()`
   Returns how many **bytes** are being used by live tensors right now.
   - The allocator rounds every block up to a multiple of 512 bytes, so the value can be slightly larger than `numel * element_size`.

6. `torch.cuda.memory_reserved()`
   Returns how many **bytes** PyTorch has reserved from the GPU (in use + cached). It is always >= `memory_allocated()`, and it is the number shown in `nvidia-smi`.
   - PyTorch does not take memory one tensor at a time: it requests large **segments** from the GPU (`cudaMalloc`) and splits them into **blocks** for tensors.
   - Analogy: the GPU is a street, a segment is a plot of land you rent, and a block is a lot inside that plot. Your tensors live in the lots. Renting is slow, so PyTorch rents a bit more than needed and reuses the free lots.

7. `torch.cuda.empty_cache()`
   Returns to the GPU the segments that are reserved but **completely empty** (no live tensor inside).
   - When a tensor dies (`del`, out of scope, reassigned), only its **block** is freed. The segment stays rented and keeps occupying VRAM until `empty_cache()` is called or the process (Jupyter kernel) dies.
   - A single live tensor keeps its whole segment from being returned.
   - Does not touch live tensors and does not speed up your code. Use it to free VRAM for other processes or between big phases.

8. `torch.cuda.synchronize()`
   Makes the CPU **wait** until the GPU finishes everything that was queued.
   - GPU operations are **asynchronous**: the CPU only enqueues the work (the kernel) and moves on to the next line, even if the GPU has not finished.
   - Analogy: ordering a pizza by phone. Hanging up does not mean the pizza is ready. `synchronize()` is waiting for it to arrive.
   - Use it mainly for **benchmarking** (call it before stopping the timer) and for debugging CUDA errors.
   - Operations that bring data back to the CPU (`print(y)`, `.item()`, `.cpu()`, `.numpy()`) synchronize automatically.




# 1. Cuda ARGS

In [1]:
import torch
import torch.cuda as cuda

# See if cuda its avaliable in you computer, if you have an GPU that you can use
print(f'Cuda its avaliable?: {torch.cuda.is_available()}')

# If you have more than 1 GPU, you can see which GPU your cuda are using
print(f'Actual cuda device: {torch.cuda.current_device()} ')

# You can see the name of the GPUS (in this case GPU 0)
print(f'Name of the first GPU: {torch.cuda.get_device_name(0)}')

# You can see how many GPUs exist in you computer
print(f'How many GPUS exist?: {torch.cuda.device_count()}')

Cuda its avaliable?: True
Actual cuda device: 0 
Name of the first GPU: NVIDIA GeForce GTX 1650
How many GPUS exist?: 1


# 2. Memory Allocated

In [2]:
tensor = torch.randn(100, 100, dtype = torch.float32, device = 'cuda')
print(f'First allocated in the memory: {cuda.memory_allocated()}') # 100 x 100 = 10.000 flotas * 4 bytes per byte = 40.000 BYTES + some butes

tensor2 = torch.randn(100, 1000, dtype = torch.float16, device = 'cuda')
print(f'Second allocated in the memory: {cuda.memory_allocated()}') 
# TENSOR 1:  100 x 100 = 10.000 flotas * 4 bytes per number = 40.000 BYTES + some deterministic
# TENSOR 2: 100 X 1000 = 100.000 floats * 2 bytes per number= 200.000 BYTES + some deterministic
# If sum both = 200.000 + 40.000= 240.000 BYTES + some deterministic noise

First allocated in the memory: 40448
Second allocated in the memory: 240640


# 3. GPU Asynchrone

In [3]:
x = torch.randn(10000, 10000, device="cuda")
y = x @ x
print("hi")               #  0.1s

# The code finished, but the GPU is still processing the computation

hi


In [4]:
x = torch.randn(10000, 10000, device="cuda")
y = x @ x
print("hi")               # 0.0s
torch.cuda.synchronize() 
print("finish")           # 3.7s


hi
finish


# 4. Memory Reserved 

In [5]:
print(f'Actual size of memory reserved: {cuda.memory_reserved()}')
print(f'Actual size of memory used: {cuda.memory_allocated()}')

if cuda.memory_reserved() > cuda.memory_allocated():
    cuda.empty_cache()
print(f'=' * 50)

print(f'Actual size of memory reserved after the cache empty: {cuda.memory_reserved()}')
print(f'Actual size of memory used after the cache empty: {cuda.memory_allocated()}')

Actual size of memory reserved: 1224736768
Actual size of memory used: 809872384
Actual size of memory reserved after the cache empty: 824180736
Actual size of memory used after the cache empty: 809872384
